# iSCORS-Net — Fast Runner

**Workflow:**
1. Run **Setup** — clones the repo and installs deps.
2. Run **Train** — internal learning on the test video.
3. Run **Results** — inline visualisation.
4. Run **Download** — saves `results_<VERSION>.zip` to your machine.

---

In [ ]:
VERSION = 'v3.0'
print(f'iSCORS-Net {VERSION}')

In [ ]:
import os

REPO   = 'https://github.com/breezy90126/iscors-net.git'
BRANCH = 'claude/beautiful-volta-gFUot'

if not os.path.isdir('iscors-net'):
    !git clone --depth 1 -b {BRANCH} {REPO}
else:
    !git -C iscors-net pull

os.chdir('iscors-net')
print('Working dir:', os.getcwd())

!pip install -q -r requirements.txt scipy
print('Dependencies installed.')

In [ ]:
import os

# Always regenerate — ensures new background design (near-static) is used
video_path = './data/test_synthetic_cell.tif'
if os.path.exists(video_path):
    os.remove(video_path)
    print('Removed old video.')

os.makedirs('./data', exist_ok=True)
!python utils/generate_test_video.py

In [ ]:
!python train_phys_recon.py

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import glob, os

result_files = sorted(glob.glob('./result/*.png'))
print(f'Result images ({len(result_files)}):', [os.path.basename(f) for f in result_files])

for path in result_files:
    img = mpimg.imread(path)
    plt.figure(figsize=(10, 4))
    plt.imshow(img)
    plt.axis('off')
    plt.title(os.path.basename(path))
    plt.tight_layout()
    plt.show()

In [ ]:
import zipfile, os, glob
from google.colab import files

zip_name = f'results_{VERSION}.zip'

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pattern in [f'./result/*{VERSION}*', f'./checkpoint/*{VERSION}*']:
        for path in glob.glob(pattern):
            zf.write(path, os.path.relpath(path, '.'))
    for path in glob.glob('./result/*.png'):
        arcname = os.path.relpath(path, '.')
        if arcname not in zf.namelist():
            zf.write(path, arcname)

print(f'Created {zip_name}  ({os.path.getsize(zip_name)/1024:.1f} KB)')
files.download(zip_name)

---

## Real Cell Analysis

**Workflow:**
1. **real-setup** — Mount Google Drive, unzip `large_file.zip`, locate files.
2. **real-mat** — Decode `Output_iSCORS_map.mat` → TIF reference maps (not fed to model).
3. **real-preprocess** — Normalise video: ÷ temporal median → ÷ per-frame Gaussian (σ=4).
4. **real-inference** — Run trained model on preprocessed video.
5. **real-compare** — Side-by-side: model predictions vs traditional iSCORS reference.


In [ ]:
# ── Real Cell Analysis: Mount Drive & Extract ─────────────────────────────
from google.colab import drive
import zipfile, os

drive.mount('/content/drive', force_remount=False)

ZIP_PATH    = '/content/drive/MyDrive/iscors_test/large_file.zip'
EXTRACT_DIR = '/content/real_data'
SAVE_DIR    = '/content/drive/MyDrive/iscors_test'

os.makedirs(EXTRACT_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,    exist_ok=True)

print(f'Extracting {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Extraction done.')

def find_file(root, name):
    for dirpath, _, files in os.walk(root):
        if name in files:
            return os.path.join(dirpath, name)
    return None

VIDEO_PATH = find_file(EXTRACT_DIR, 'COBRI_rarw_video.tif')
MAT_PATH   = find_file(EXTRACT_DIR, 'Output_iSCORS_map.mat')

print(f'Video : {VIDEO_PATH}')
print(f'MAT   : {MAT_PATH}')
assert VIDEO_PATH, 'COBRI_rarw_video.tif not found in zip!'
assert MAT_PATH,   'Output_iSCORS_map.mat not found in zip!'


In [ ]:
# ── Decode Output_iSCORS_map.mat → TIF (reference only, not fed to model) ──
import numpy as np
import tifffile

# Try scipy.io (MATLAB v5/v7); fall back to h5py (v7.3 / HDF5)
gamma_ref = alpha_ref = None
try:
    import scipy.io as sio
    mat = sio.loadmat(MAT_PATH)
    data = {k: v for k, v in mat.items() if not k.startswith('_')}
    print('Loaded via scipy.io.  Fields:', list(data.keys()))
    _h5 = False
except Exception as e:
    print(f'scipy.io failed ({e}), trying h5py ...')
    import h5py
    _h5_file = h5py.File(MAT_PATH, 'r')
    data = {k: _h5_file[k] for k in _h5_file}
    print('Loaded via h5py.  Fields:', list(data.keys()))
    _h5 = True

def _find(d, *names):
    lmap = {k.lower(): k for k in d}
    for n in names:
        if n.lower() in lmap:
            return np.array(d[lmap[n.lower()]]).squeeze().astype(np.float32)
    return None

gamma_ref = _find(data, 'gamma', 'Gamma', 'gamma_map', 'D')
alpha_ref = _find(data, 'alpha', 'Alpha', 'alpha_map', 'beta', 'anomalous_exp')

if _h5:
    _h5_file.close()

for name, arr in [('gamma_ref', gamma_ref), ('alpha_ref', alpha_ref)]:
    if arr is not None:
        print(f'{name}: shape={arr.shape}  range=[{arr.min():.4f}, {arr.max():.4f}]')
    else:
        print(f'{name}: field not auto-detected — inspect data keys above and set manually')

# Save reference TIFs to Drive
if gamma_ref is not None:
    p = os.path.join(SAVE_DIR, 'iscors_gamma_ref.tif')
    tifffile.imwrite(p, gamma_ref)
    print(f'Saved -> {p}')
if alpha_ref is not None:
    p = os.path.join(SAVE_DIR, 'iscors_alpha_ref.tif')
    tifffile.imwrite(p, alpha_ref)
    print(f'Saved -> {p}')


In [ ]:
# ── Preprocess Real Video ──────────────────────────────────────────────────
# Step 1 — flat-field:  divide each frame by per-pixel temporal median
# Step 2 — background:  divide each frame by its own Gaussian-smoothed image (σ=4)
import numpy as np
import tifffile
from scipy.ndimage import gaussian_filter

print(f'Loading {VIDEO_PATH} ...')
video_raw = tifffile.imread(VIDEO_PATH).astype(np.float32)
T, H, W = video_raw.shape
print(f'Raw:  T={T}  H={H}  W={W}  '
      f'range=[{video_raw.min():.1f}, {video_raw.max():.1f}]')

# Step 1: flat-field correction
print('\nStep 1: flat-field (÷ temporal median) ...')
median_xy = np.median(video_raw, axis=0)                    # (H, W)
video_ff  = video_raw / (median_xy[np.newaxis] + 1e-10)    # (T, H, W) ≈ 1.0 background
print(f'  mean={video_ff.mean():.4f}  std={video_ff.std():.6f}')

# Step 2: per-frame Gaussian background division
print('Step 2: Gaussian background division (σ=4, per frame) ...')
video_proc = np.empty_like(video_ff)
for t in range(T):
    bg = gaussian_filter(video_ff[t], sigma=4)
    video_proc[t] = video_ff[t] / (bg + 1e-10)
    if t % max(1, T // 5) == 0:
        print(f'  frame {t:4d}/{T}')

print(f'\nProcessed: mean={video_proc.mean():.4f}  std={video_proc.std():.6f}  '
      f'range=[{video_proc.min():.4f}, {video_proc.max():.4f}]')
print('Preprocessing done.')


In [ ]:
# ── Real Video Inference ───────────────────────────────────────────────────
import torch
from datasets.phys_recon_dataset import PhysReconDataset
from models.pissl_tau_encoder    import PISSLTauEncoder

RECON_TAUS = (1, 2, 4, 8, 16, 32, 48, 64, 96, 128)
CKPT_PATH  = f'./checkpoint/pissl_phys_recon_{VERSION}.pth'
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Compute G_empirical from preprocessed video (mode=eval → no blind-spot)
print('Computing G_empirical on real video ...')
real_ds = PhysReconDataset(
    video_tensor   = video_proc,
    recon_taus     = RECON_TAUS,
    patch_size     = 64,
    mode           = 'eval',
)

# Load trained model
print(f'Loading checkpoint: {CKPT_PATH}')
model = PISSLTauEncoder(recon_taus=RECON_TAUS, predict_amplitude=False).to(device)
model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
model.eval()

# Pad to multiple-of-8 if needed (3× MaxPool2d in encoder)
full_input = real_ds[0]                                    # (K, H, W)
K, Hv, Wv = full_input.shape
pad_h = (8 - Hv % 8) % 8
pad_w = (8 - Wv % 8) % 8
if pad_h or pad_w:
    import torch.nn.functional as F
    full_input = F.pad(full_input, (0, pad_w, 0, pad_h))
    print(f'Padded input: {Hv}x{Wv} -> {Hv+pad_h}x{Wv+pad_w}')

full_input = full_input.unsqueeze(0).to(device)            # (1, K, H', W')
print(f'Input: {full_input.shape}')

with torch.no_grad():
    preds_real = model(full_input)                         # (1, 2, H', W')

# Crop padding back and apply cell mask
gamma_pred = preds_real[0, 0].cpu().numpy()[:Hv, :Wv]
alpha_pred = preds_real[0, 1].cpu().numpy()[:Hv, :Wv]
bg_mask    = ~real_ds.cell_mask
gamma_pred[bg_mask] = float('nan')
alpha_pred[bg_mask] = float('nan')

cell = real_ds.cell_mask
print(f'γ cell: [{np.nanmin(gamma_pred):.3f}, {np.nanmax(gamma_pred):.3f}]')
print(f'α cell: [{np.nanmin(alpha_pred):.3f}, {np.nanmax(alpha_pred):.3f}]')
print('Inference done.')


In [ ]:
# ── Comparison: Model Predictions vs Traditional iSCORS Reference ──────────
import matplotlib.pyplot as plt
import numpy as np
import tifffile, os

def _match_shape(arr, target_shape):
    if arr.shape == target_shape:
        return arr
    from skimage.transform import resize
    return resize(arr, target_shape, preserve_range=True).astype(np.float32)

# Build panel list: (title, data, cmap, vmin, vmax)
panels = [
    (f'Model γ [{VERSION}]', gamma_pred, 'magma',   0,    1.0),
    (f'Model α [{VERSION}]', alpha_pred, 'viridis', 0.5,  2.0),
]
if gamma_ref is not None:
    gr = _match_shape(gamma_ref, gamma_pred.shape)
    panels.append(('iSCORS γ (ref)', gr, 'magma',
                   np.nanpercentile(gr, 2), np.nanpercentile(gr, 98)))
if alpha_ref is not None:
    ar = _match_shape(alpha_ref, alpha_pred.shape)
    panels.append(('iSCORS α (ref)', ar, 'viridis',
                   np.nanpercentile(ar, 2), np.nanpercentile(ar, 98)))

fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
if len(panels) == 1:
    axes = [axes]
for ax, (title, data, cmap, vmin, vmax) in zip(axes, panels):
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.axis('off')

fig.suptitle('Real Cell Analysis', fontsize=13, y=1.02)
fig.tight_layout()
plt.show()

# Save to Drive
out_fig = os.path.join(SAVE_DIR, f'real_inference_{VERSION}.png')
fig.savefig(out_fig, dpi=120, bbox_inches='tight')
print(f'Saved figure  -> {out_fig}')

gamma_save = np.where(real_ds.cell_mask, gamma_pred, 0).astype(np.float32)
alpha_save = np.where(real_ds.cell_mask, alpha_pred, 0).astype(np.float32)
tifffile.imwrite(os.path.join(SAVE_DIR, f'model_gamma_{VERSION}.tif'), gamma_save)
tifffile.imwrite(os.path.join(SAVE_DIR, f'model_alpha_{VERSION}.tif'), alpha_save)
print(f'Saved model γ -> {SAVE_DIR}/model_gamma_{VERSION}.tif')
print(f'Saved model α -> {SAVE_DIR}/model_alpha_{VERSION}.tif')
